In [ ]:
import os
import warnings
import boto3
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

#### Functions

In [ ]:
# feature engineer
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, list_drop_cols=None):
        self.list_drop_cols = list_drop_cols or ['loan_id', 'charged_off_amount', 'paid_interest_amount', 'origination_date']

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        # engineer age from dob
        if 'dob' in df.columns:
            df['dob'] = pd.to_datetime(df['dob'])
            df['int_age'] = ((pd.Timestamp.now() - df['dob']).dt.days / 365.25).astype(int)
            df = df.drop(columns=['dob'])
        # engineer payment to income ratio
        if 'loan_amount' in df.columns and 'stated_income' in df.columns:
            df['flt_payment_to_income'] = df['loan_amount'] / df['stated_income'].replace(0, np.nan)
        # drop columns
        list_cols_to_drop = [col for col in self.list_drop_cols if col in df.columns]
        df = df.drop(columns=list_cols_to_drop)
        return df

In [ ]:
# plot missing before and after preprocessing
def plot_missing_before_after(df_before, df_after, str_filename='output/missing_before_after.png'):
    ser_before = df_before.isnull().mean()
    ser_after = df_after.isnull().mean()
    # only show columns that had missing values before
    ser_before = ser_before[ser_before > 0].sort_values(ascending=True)
    if len(ser_before) == 0:
        print('No missing values found before preprocessing.')
        return
    ser_after = ser_after.reindex(ser_before.index).fillna(0)
    # plot
    list_cols = ser_before.index.tolist()
    arr_y = np.arange(len(list_cols))
    flt_bar_height = 0.35
    fig, ax = plt.subplots(figsize=(10, max(5, len(list_cols) * 0.5)))
    ax.barh(arr_y + flt_bar_height / 2, ser_before.values, flt_bar_height, label='Before', color='salmon', edgecolor='black')
    ax.barh(arr_y - flt_bar_height / 2, ser_after.values, flt_bar_height, label='After', color='steelblue', edgecolor='black')
    ax.set_yticks(arr_y)
    ax.set_yticklabels(list_cols)
    ax.set_title('Proportion Missing Before and After Preprocessing', fontsize=16)
    ax.set_xlabel('Proportion Missing', fontsize=12)
    ax.legend()
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()

#### Constants

In [ ]:
# bucket
str_bucket = os.getcwd().split('/')[4].replace('_', '-')
print(f'Bucket: {str_bucket}')

# step
str_step = os.getcwd().split('/')[-1]
print(f'Step: {str_step}')

# s3 input path
str_s3_input = f's3://{str_bucket}/02_data_split'
print(f'S3 Input: {str_s3_input}')

# target
str_target = 'default_12m'
print(f'Target: {str_target}')

# output directory
os.makedirs('output', exist_ok=True)

#### Read Data

In [ ]:
# read data
df_train = pd.read_parquet(f'{str_s3_input}/df_train.parquet')
df_valid = pd.read_parquet(f'{str_s3_input}/df_valid.parquet')
df_test = pd.read_parquet(f'{str_s3_input}/df_test.parquet')

# print shapes
for str_name, df_split in [('Train', df_train), ('Validation', df_valid), ('Test', df_test)]:
    print(f'{str_name}: {df_split.shape}')

#### Define Pipeline

In [ ]:
# apply feature engineering to determine column names
feature_engineer = FeatureEngineer()
df_train_fe = feature_engineer.transform(df_train)

# separate target
list_feature_cols = [col for col in df_train_fe.columns if col != str_target]

# identify numeric and categorical columns
list_numeric_cols = df_train_fe[list_feature_cols].select_dtypes(include=[np.number]).columns.tolist()
list_categorical_cols = df_train_fe[list_feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numeric features ({len(list_numeric_cols)}): {list_numeric_cols}')
print(f'Categorical features ({len(list_categorical_cols)}): {list_categorical_cols}')

In [ ]:
# define column transformer
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

column_transformer = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, list_numeric_cols),
        ('cat', categorical_transformer, list_categorical_cols),
    ],
)

# full pipeline
pipeline = Pipeline(steps=[
    ('feature_engineer', FeatureEngineer()),
    ('column_transformer', column_transformer),
])

print('Pipeline defined successfully')
print(pipeline)

#### Fit and Transform

In [ ]:
warnings.filterwarnings('ignore')

# fit on training data ONLY
pipeline.fit(df_train)

# transform all splits
list_col_names = list_numeric_cols + list_categorical_cols

arr_train = pipeline.transform(df_train)
arr_valid = pipeline.transform(df_valid)
arr_test = pipeline.transform(df_test)

# reconstruct DataFrames
df_train_clean = pd.DataFrame(arr_train, columns=list_col_names)
df_train_clean[str_target] = df_train[str_target].values

df_valid_clean = pd.DataFrame(arr_valid, columns=list_col_names)
df_valid_clean[str_target] = df_valid[str_target].values

df_test_clean = pd.DataFrame(arr_test, columns=list_col_names)
df_test_clean[str_target] = df_test[str_target].values

warnings.filterwarnings('default')

# print shapes
for str_name, df_split in [('Train', df_train_clean), ('Validation', df_valid_clean), ('Test', df_test_clean)]:
    print(f'{str_name}: {df_split.shape}')

df_train_clean.head()

#### Missing Values Before and After

In [ ]:
plot_missing_before_after(df_train_fe, df_train_clean)

#### Save